# Seminar 09: Metric Learning and Visual Search with Image Embeddings

**Student Version**

Goals for today:
- Inspect image embeddings and understand why normalization matters
- Build top-k visual search with cosine similarity
- Evaluate retrieval with qualitative examples and Recall@K
- Compare same-class and different-class similarity distributions
- Understand triplet margin loss without running a long training loop


## 0. Setup

In [ ]:
import math
import random
import time
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Dataset
import torchvision
from torchvision import transforms
from torchvision.models import resnet18, ResNet18_Weights
import matplotlib.pyplot as plt


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


seed_everything(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device


### Config

In [ ]:
CFG = {
    'seed': 42,
    'train_n': 2500,
    'gallery_n': 2500,
    'query_n': 200,
    'batch_size': 128,
    'top_k': 8,
    'finetune_epochs': 1,
    'finetune_lr': 1e-3,
}
CFG


### Helpers

In [ ]:
CIFAR10_CLASS_NAMES = [
    'airplane', 'automobile', 'bird', 'cat', 'deer',
    'dog', 'frog', 'horse', 'ship', 'truck'
]


def print_shape(name, x):
    if x is None:
        print(f'{name}: not filled yet')
        return
    print(f'{name}: shape={tuple(x.shape)} dtype={x.dtype}')


def show_images(images, labels=None, scores=None, matches=None, class_names=None, images_per_row=6, figsize=(12, 4), title=None):
    images = images.detach().cpu()
    n_images = images.shape[0]
    n_rows = int(math.ceil(n_images / images_per_row))
    plt.figure(figsize=figsize)
    if title is not None:
        plt.suptitle(title)
    for i in range(n_images):
        plt.subplot(n_rows, images_per_row, i + 1)
        img = images[i].permute(1, 2, 0).numpy()
        plt.imshow(np.clip(img, 0, 1))
        plt.axis('off')
        text = []
        if labels is not None:
            y = int(labels[i])
            text.append(class_names[y] if class_names is not None else str(y))
        if scores is not None:
            text.append(f'{float(scores[i]):.3f}')
        if matches is not None:
            text.append('match' if bool(matches[i]) else 'miss')
        if text:
            plt.title('\n'.join(text), fontsize=8)
    plt.tight_layout()
    plt.show()


def l2_normalize(x, eps=1e-12):
    return x / (x.norm(dim=-1, keepdim=True) + eps)


def cosine_similarity_matrix(query_emb, gallery_emb):
    # query_emb: [Nq, D], gallery_emb: [Ng, D]
    return query_emb @ gallery_emb.T


class RetrievalCIFAR10(Dataset):
    def __init__(self, base_dataset, indices, model_transform, display_transform):
        self.base_dataset = base_dataset
        self.indices = list(indices)
        self.model_transform = model_transform
        self.display_transform = display_transform

    def __len__(self):
        return len(self.indices)

    def __getitem__(self, idx):
        image, label = self.base_dataset[self.indices[idx]]
        model_image = self.model_transform(image)
        display_image = self.display_transform(image)
        return model_image, label, display_image


@torch.no_grad()
def extract_embeddings(model, loader):
    model.eval()
    all_emb, all_y, all_x = [], [], []
    for xb, yb, x_display in loader:
        xb = xb.to(device)
        emb = model(xb)
        emb = l2_normalize(emb)
        all_emb.append(emb.cpu())
        all_y.append(yb.cpu())
        all_x.append(x_display.cpu())
    return torch.cat(all_emb), torch.cat(all_y), torch.cat(all_x)


## 1. Provided Infrastructure

We will use a pretrained ResNet18 as an image embedder.

This part is provided because the new ideas today are retrieval and metric learning, not repeating the transfer-learning setup from Seminar 7.


In [ ]:
weights = ResNet18_Weights.DEFAULT
model_transform = weights.transforms()
display_transform = transforms.ToTensor()

base_train = torchvision.datasets.CIFAR10(root='./data', train=True, download=True)
perm = torch.randperm(len(base_train))

train_idx = perm[:CFG['train_n']].tolist()
gallery_start = CFG['train_n']
gallery_end = gallery_start + CFG['gallery_n']
query_end = gallery_end + CFG['query_n']
gallery_idx = perm[gallery_start:gallery_end].tolist()
query_idx = perm[gallery_end:query_end].tolist()

train_ds = RetrievalCIFAR10(base_train, train_idx, model_transform, display_transform)
gallery_ds = RetrievalCIFAR10(base_train, gallery_idx, model_transform, display_transform)
query_ds = RetrievalCIFAR10(base_train, query_idx, model_transform, display_transform)

num_workers = 2
pin_memory = torch.cuda.is_available()
train_loader = DataLoader(train_ds, batch_size=CFG['batch_size'], shuffle=True, num_workers=num_workers, pin_memory=pin_memory)
gallery_loader = DataLoader(gallery_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=num_workers, pin_memory=pin_memory)
query_loader = DataLoader(query_ds, batch_size=CFG['batch_size'], shuffle=False, num_workers=num_workers, pin_memory=pin_memory)

xb_model, yb, xb_display = next(iter(gallery_loader))
print_shape('xb_model', xb_model)
print_shape('yb', yb)
print_shape('xb_display', xb_display)
show_images(xb_display[:12], labels=yb[:12], class_names=CIFAR10_CLASS_NAMES, images_per_row=6, figsize=(12, 5), title='Gallery examples')


In [ ]:
def build_resnet18_embedder(weights):
    model = resnet18(weights=weights)
    model.fc = nn.Identity()
    model = model.to(device)
    model.eval()
    return model


embedder = build_resnet18_embedder(weights)

with torch.no_grad():
    sample_raw_emb = embedder(xb_model[:8].to(device)).cpu()
print_shape('sample_raw_emb', sample_raw_emb)

print('Extracting gallery/query embeddings...')
t0 = time.time()
gallery_emb, gallery_y, gallery_x = extract_embeddings(embedder, gallery_loader)
query_emb, query_y, query_x = extract_embeddings(embedder, query_loader)
print('elapsed sec:', round(time.time() - t0, 1))
print_shape('gallery_emb', gallery_emb)
print_shape('query_emb', query_emb)


## 2. Embedding Sanity Checks

The pretrained backbone outputs feature vectors. Before search, we need to understand their shape and norms.

### Exercise 1
Inspect one batch of raw embeddings, normalize them, and compare dot product with cosine similarity.

Useful contracts:
- `embedder(xb)` returns `[B, D]` embeddings because the final classifier was replaced with `nn.Identity()`.
- `x.norm(dim=-1)` computes one vector norm per image.
- After L2 normalization, dot product equals cosine similarity.


In [ ]:
# Exercise 1

sample_x = xb_model[:16].to(device)

with torch.no_grad():
    raw_emb = None

normed_emb = None
raw_norms = None
normed_norms = None

dot_sims_raw = None
dot_sims_normed = None
cosine_sims_raw = None

print_shape('raw_emb', raw_emb)
print('raw norms, first 5:', raw_norms[:5] if raw_norms is not None else None)
print('normalized norms, first 5:', normed_norms[:5] if normed_norms is not None else None)
print('raw dot similarities, first row:', dot_sims_raw[0, :5] if dot_sims_raw is not None else None)
print('normalized dot similarities, first row:', dot_sims_normed[0, :5] if dot_sims_normed is not None else None)


### Checks (Exercise 1)

In [ ]:
assert raw_emb.ndim == 2
assert raw_emb.shape[0] == 16
assert normed_emb.shape == raw_emb.shape
assert raw_norms.shape == (16,)
assert normed_norms.shape == (16,)
assert torch.allclose(normed_norms, torch.ones_like(normed_norms), atol=1e-5)
assert dot_sims_raw.shape == (16, 16)
assert dot_sims_normed.shape == (16, 16)
assert cosine_sims_raw.shape == (16, 16)
assert torch.allclose(dot_sims_normed, cosine_sims_raw, atol=1e-5)
print('Exercise 1 passed.')


## 3. Top-K Visual Search

A visual search system compares every query embedding with every gallery embedding.

### Exercise 2
Build the similarity matrix and retrieve the top-k gallery images for every query.

Useful contracts:
- `query_emb`: `[Nq, D]`
- `gallery_emb`: `[Ng, D]`
- `similarity_matrix = query_emb @ gallery_emb.T`: `[Nq, Ng]`
- `torch.topk(similarity_matrix, k, dim=1)` ranks gallery images for each query.
- `dim=0` would rank queries for each gallery image, which is the wrong axis for search.


In [ ]:
# Exercise 2

top_k = CFG['top_k']

similarity_matrix = None
top_scores = None
top_idx = None

wrong_axis_scores = None
wrong_axis_idx = None

print_shape('similarity_matrix', similarity_matrix)
print_shape('top_scores', top_scores)
print_shape('top_idx', top_idx)
print('first query top labels:', [CIFAR10_CLASS_NAMES[int(y)] for y in gallery_y[top_idx[0]]] if top_idx is not None else None)


### Checks (Exercise 2)

In [ ]:
assert similarity_matrix.shape == (len(query_y), len(gallery_y))
assert top_scores.shape == (len(query_y), top_k)
assert top_idx.shape == (len(query_y), top_k)
assert top_idx.dtype == torch.long
assert wrong_axis_idx.shape == (top_k, len(gallery_y))
assert torch.all(top_idx >= 0) and torch.all(top_idx < len(gallery_y))
print('Exercise 2 passed.')


## 4. Retrieval Visualization and Failure Analysis

Retrieval quality is not only a number. We should inspect what the embedding space considers similar.

### Exercise 3
Implement a helper that shows a query image and its retrieved neighbors. Include labels, similarity scores, and match/miss markers.

Function contract:
- `query_index`: which query image to inspect.
- `query_x`: display images for the query set, shape `[Nq, C, H, W]`.
- `query_y`: labels for query images, shape `[Nq]`.
- `gallery_x`: display images for the gallery/search database, shape `[Ng, C, H, W]`.
- `gallery_y`: labels for gallery images, shape `[Ng]`.
- `top_scores`: similarity scores returned by retrieval, shape `[Nq, top_k]`.
- `top_idx`: gallery indices returned by retrieval, shape `[Nq, top_k]`.
- `class_names`: list mapping class ids to readable labels.
- `k`: how many retrieved images to show.

Useful contracts:
- `query_x[query_index:query_index+1]` keeps the batch dimension.
- `gallery_x[top_idx[query_index, :k]]` selects the retrieved gallery images.
- A match means `gallery_y[retrieved_index] == query_y[query_index]`.


In [ ]:
# Exercise 3

def show_retrieval_result(
    query_index,
    query_x,
    query_y,
    gallery_x,
    gallery_y,
    top_scores,
    top_idx,
    class_names,
    k=8,
):
    retrieved_idx = None
    retrieved_scores = None
    retrieved_labels = None
    matches = None

    # Show the query first, then the retrieved gallery images.
    # Use show_images(...). Keep this function reusable for different query/gallery results.
    query_label = class_names[int(query_y[query_index])]
    show_images(
        query_x[query_index:query_index + 1],
        labels=query_y[query_index:query_index + 1],
        class_names=class_names,
        images_per_row=1,
        figsize=(2, 2),
        title=f'Query {query_index}: {query_label}'
    )
    show_images(
        gallery_x[retrieved_idx],
        labels=retrieved_labels,
        scores=retrieved_scores,
        matches=matches,
        class_names=class_names,
        images_per_row=k,
        figsize=(14, 3),
        title='Retrieved images'
    )
    
    return matches

for query_index in [0, 1, 2, 3]:
    matches = show_retrieval_result(
        query_index=query_index,
        query_x=query_x,
        query_y=query_y,
        gallery_x=gallery_x,
        gallery_y=gallery_y,
        top_scores=top_scores,
        top_idx=top_idx,
        class_names=CIFAR10_CLASS_NAMES,
        k=CFG['top_k'],
    )
    print('query', query_index, 'matches:', matches.tolist() if matches is not None else None)


### Checks (Exercise 3)

In [ ]:
test_matches = show_retrieval_result(
    query_index=0,
    query_x=query_x,
    query_y=query_y,
    gallery_x=gallery_x,
    gallery_y=gallery_y,
    top_scores=top_scores,
    top_idx=top_idx,
    class_names=CIFAR10_CLASS_NAMES,
    k=CFG['top_k'],
)
assert test_matches.shape == (CFG['top_k'],)
assert test_matches.dtype == torch.bool
print('Exercise 3 passed.')


## 5. Recall@K

Recall@K asks whether at least one relevant gallery image appears in the first K retrieved results.

For CIFAR-10, we will use same class label as the relevance signal.

### Exercise 4
Implement `recall_at_k` and report `Recall@1`, `Recall@5`, and `Recall@8`.

Useful contracts:
- `top_indices[:, :k]` keeps the first `k` retrieved gallery indices for each query.
- `gallery_labels[top_indices[:, :k]]` gives retrieved labels with shape `[Nq, k]`.
- A query succeeds if any retrieved label equals its query label.


In [ ]:
# Exercise 4

def recall_at_k(query_labels, gallery_labels, top_indices, k):
    retrieved_labels = None
    matches = None
    success = None
    return None

recall_1 = None
recall_5 = None
recall_8 = None

print('Recall@1:', recall_1)
print('Recall@5:', recall_5)
print('Recall@8:', recall_8)


### Checks (Exercise 4)

In [ ]:
assert 0.0 <= recall_1 <= 1.0
assert 0.0 <= recall_5 <= 1.0
assert 0.0 <= recall_8 <= 1.0
assert recall_1 <= recall_5 + 1e-8
assert recall_5 <= recall_8 + 1e-8
print('Exercise 4 passed.')


## 6. Same-Class vs Different-Class Similarity

A useful sanity check: same-class pairs should usually have higher similarity than different-class pairs.

### Exercise 5
Sample balanced same-class and different-class pairs from the gallery embeddings, then compare the similarity distributions.

Useful contracts:
- Since embeddings are L2-normalized, cosine similarity is `(emb_i * emb_j).sum()`.
- Use `labels == labels[i]` to find positive candidates.
- Use `labels != labels[i]` to find negative candidates.
- `torch.where(condition)[0]` returns the indices where the condition is true.


In [ ]:
# Exercise 5

def sample_balanced_pair_similarities(embeddings, labels, n_pairs_each=500):
    same_sims = []
    diff_sims = []
    n = embeddings.shape[0]

    for _ in range(n_pairs_each):
        i = torch.randint(0, n, (1,)).item()
        same_candidates = None
        j_same = None
        same_sims.append(None)

        i = torch.randint(0, n, (1,)).item()
        diff_candidates = None
        j_diff = None
        diff_sims.append(None)

    return torch.tensor(same_sims), torch.tensor(diff_sims)

same_sims, diff_sims = sample_balanced_pair_similarities(gallery_emb, gallery_y, n_pairs_each=500)

same_mean = None
diff_mean = None

print('mean same-class similarity     :', same_mean)
print('mean different-class similarity:', diff_mean)

# Plot histograms after the values are computed.
# plt.figure(figsize=(8, 4))
# plt.hist(same_sims.numpy(), bins=30, alpha=0.7, label='same class')
# plt.hist(diff_sims.numpy(), bins=30, alpha=0.7, label='different class')
# plt.xlabel('cosine similarity')
# plt.ylabel('count')
# plt.legend()
# plt.show()


### Checks (Exercise 5)

In [ ]:
assert same_sims.ndim == 1 and diff_sims.ndim == 1
assert len(same_sims) == 500
assert len(diff_sims) == 500
assert torch.isfinite(same_sims).all()
assert torch.isfinite(diff_sims).all()
assert -1.0 <= same_mean <= 1.0
assert -1.0 <= diff_mean <= 1.0
print('Exercise 5 passed.')


## 7. Triplet Margin Mini-Lab

Triplet loss compares an anchor, a positive example from the same class, and a negative example from a different class.

We will compute the loss directly on embeddings, without training.

### Exercise 6
Sample triplets and compute the manual triplet margin loss:

```text
max(0, d(anchor, positive) - d(anchor, negative) + margin)
```

Useful contracts:
- `torch.where(condition)[0]` returns the indices where the condition is true.
- `torch.norm(a - b, dim=1)` computes one Euclidean distance per row.
- `torch.clamp(values, min=0)` applies the hinge part of the loss.
- Loss is zero when the negative is already farther away by at least the margin.


In [ ]:
# Exercise 6

def sample_triplet_indices(labels, n_triplets=64):
    anchors = []
    positives = []
    negatives = []
    n = len(labels)

    for _ in range(n_triplets):
        anchor = torch.randint(0, n, (1,)).item()
        positive_candidates = None
        negative_candidates = None
        positive = None
        negative = None
        anchors.append(anchor)
        positives.append(positive)
        negatives.append(negative)

    return torch.tensor(anchors), torch.tensor(positives), torch.tensor(negatives)

margin = 0.2
anchor_idx, positive_idx, negative_idx = sample_triplet_indices(gallery_y, n_triplets=64)

anchor_emb = None
positive_emb = None
negative_emb = None

d_ap = None
d_an = None
triplet_losses = None
manual_triplet_loss = None

print('mean d(anchor, positive):', d_ap.mean().item() if d_ap is not None else None)
print('mean d(anchor, negative):', d_an.mean().item() if d_an is not None else None)
print('manual triplet loss:', manual_triplet_loss)


### Checks (Exercise 6)

In [ ]:
assert anchor_idx.shape == positive_idx.shape == negative_idx.shape == (64,)
assert torch.all(gallery_y[anchor_idx] == gallery_y[positive_idx])
assert torch.all(gallery_y[anchor_idx] != gallery_y[negative_idx])
assert d_ap.shape == d_an.shape == triplet_losses.shape == (64,)
assert torch.all(triplet_losses >= 0)
assert manual_triplet_loss >= 0
print('Exercise 6 passed.')


## 8. Demo: Fine-Tuned Classifier Embeddings

If there is time, run a live demo comparing:
- pretrained ImageNet embeddings
- embeddings from a ResNet18 briefly fine-tuned as a CIFAR-10 classifier

The point is not the training loop itself. The point is to ask whether task-specific fine-tuning improves class-based retrieval metrics such as Recall@K.


## 9. Wrap-Up Questions
1. What is the difference between classifier logits and embeddings?
2. Why can raw dot product be misleading before normalization?
3. In a similarity matrix `[Nq, Ng]`, which axis should `topk` use for visual search?
4. What does Recall@K measure that top-1 accuracy does not?
5. What does triplet loss do when the negative example is already far enough away?
